<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/ace_step_1-5-customi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import shutil

# 1. Dọn dẹp sạch sẽ
if os.path.exists("Ace-Step-v1.5"):
    shutil.rmtree("Ace-Step-v1.5")

print("📥 Cloning ACE-Step v1.5 repository...")
!git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5

# 2. Vào thư mục làm việc
os.chdir("/content/Ace-Step-v1.5")
print(f"📂 Working directory: {os.getcwd()}")

# 3. Vá lỗi code gốc
print("🔧 Patching code...")
# Sửa lỗi version torch
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt
# Bật link public
!sed -i 's/share=False/share=True/g' app.py
# Chèn code tự động dọn RAM sau mỗi lần tạo nhạc
!sed -i '/yield/i \ \ \ \ \ \ \ \ import torch; torch.cuda.empty_cache(); import gc; gc.collect()' app.py

print("✅ Bước 1 Hoàn tất.")

📥 Cloning ACE-Step v1.5 repository...
Cloning into 'Ace-Step-v1.5'...
remote: Enumerating objects: 1298, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1298 (delta 0), reused 0 (delta 0), pack-reused 1295 (from 1)
Receiving objects: 100% (1298/1298), 1.57 MiB | 2.43 MiB/s, done.
Resolving deltas: 100% (789/789), done.
📂 Working directory: /content/Ace-Step-v1.5
🔧 Patching code...
✅ Bước 1 Hoàn tất.


In [2]:
import os

if os.getcwd() != "/content/Ace-Step-v1.5":
    os.chdir("/content/Ace-Step-v1.5")

print("📦 Installing Dependencies (Lite Mode)...")

# Chỉ cài những thứ cần thiết, bỏ qua flash-attn nặng nề gây treo máy
!pip install --no-cache-dir -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install --no-cache-dir ffmpeg-python
!apt-get install -y ffmpeg

# Xóa cache pip để giải phóng RAM ngay lập tức
!rm -rf /root/.cache/pip

print("✅ Bước 2 Hoàn tất. Dependencies Ready.")

📦 Installing Dependencies (Lite Mode)...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu128
Ignoring torch: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchaudio: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchvision: markers 'sys_platform == "win32"' don't match your environment
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "win32" and python_version == "3.11" and platform_machine == "AMD64"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "linux" and python_version == "3.11"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 26.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 294.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 197.0 MB/s eta 0:00:0

^C
✅ Bước 2 Hoàn tất. Dependencies Ready.


In [ ]:
import os
import torch
import gc

if os.getcwd() != "/content/Ace-Step-v1.5":
    os.chdir("/content/Ace-Step-v1.5")

# --- PHẦN 1: HACK RAM (CHỐNG SẬP) ---
print("💾 Đang tạo RAM ảo (Swap Memory) từ ổ cứng...")
# Lấy 10GB ổ cứng đắp vào làm RAM
!sudo fallocate -l 10G /swapfile
!sudo chmod 600 /swapfile
!sudo mkswap /swapfile
!sudo swapon /swapfile
print("✅ Đã kích hoạt thêm 10GB RAM ảo. Giờ bạn có tổng cộng ~23GB RAM!")

# --- PHẦN 2: CẤU HÌNH LITE ---
# Tắt model phụ (tiết kiệm 4GB VRAM GPU)
os.environ["SERVICE_MODE_DIT_MODEL_2"] = ""
# Giảm độ dài ngữ cảnh xuống mức an toàn
os.environ["MAX_MODEL_LEN"] = "1024"
# Quản lý bộ nhớ GPU chặt chẽ
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Dọn rác lần cuối
gc.collect()
torch.cuda.empty_cache()

# --- PHẦN 3: TẢI MODEL & CHẠY ---
download_code = """
import os
import sys
from acestep.handler import AceStepHandler

print("⏳ Đang kiểm tra và tải Model (10GB)...")
current_dir = os.getcwd()
sys.path.insert(0, os.path.join(current_dir, "acestep", "third_parts", "nano-vllm"))

handler = AceStepHandler(persistent_storage_path=os.path.join(current_dir, "data"))
config_path = "acestep-v15-turbo"

try:
    # Kích hoạt tải về
    handler.initialize_service(current_dir, config_path, device='cpu', offload_to_cpu=True)
except Exception:
    pass
print("✅ Model đã sẵn sàng.")
"""

with open("dl_script.py", "w") as f:
    f.write(download_code)

print("🚀 Đang tải model...")
!python dl_script.py

print("="*60)
print("🎹 Đang khởi động Server...")
print("⚠️ Nếu thấy chữ đỏ 'nano-vllm not installed' -> KỆ NÓ, ĐỪNG LO.")
print("👉 Đợi link 'gradio.live' hiện ra và bấm vào.")
print("="*60)

!python app.py

💾 Đang tạo RAM ảo (Swap Memory) từ ổ cứng...
Setting up swapspace version 1, size = 10 GiB (10737414144 bytes)
no label, UUID=fec2e669-34e0-4dbd-8376-65982f61f345
swapon: /swapfile: swapon failed: Invalid argument
✅ Đã kích hoạt thêm 10GB RAM ảo. Giờ bạn có tổng cộng ~23GB RAM!
🚀 Đang tải model...
2026-02-14 05:14:11.220559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771046051.430946    1818 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771046051.485173    1818 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771046051.883374    1818 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin